# Customer 360 Accelerator
### Notebook 04 — Query the Data Agent

---

> **Purpose:** Send natural-language queries to the published Data Agent
> and display the results. Demonstrates the full end-to-end flow:
> CSV -> Delta Table -> Semantic Model -> Ontology -> Data Agent -> Chat.
>
> **Prerequisite:** Run notebooks `00` through `03` first.

---
## Configuration

In [ ]:
# ── Paste outputs from previous notebooks, or leave blank to auto-detect ─────
WORKSPACE_ID = ""
DATAAGENT_ID = ""

DATAAGENT_NAME   = "Customer360Agent"
LAKEHOUSE_NAME   = "Customer360Lakehouse"

In [ ]:
import sempy.fabric as fabric
import json, time

client = fabric.FabricRestClient()

# Auto-detect workspace
if not WORKSPACE_ID:
    WORKSPACE_ID = fabric.get_notebook_workspace_id()

# Auto-detect agent ID
if not DATAAGENT_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/dataAgents")
    if resp.status_code == 200:
        agent = next(
            (a for a in resp.json().get("value", []) if a.get("displayName") == DATAAGENT_NAME),
            None,
        )
        if agent:
            DATAAGENT_ID = agent["id"]
    
    # Fallback: generic items API
    if not DATAAGENT_ID:
        resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/items?type=DataAgent")
        if resp.status_code == 200:
            agent = next(
                (i for i in resp.json().get("value", []) if i.get("displayName") == DATAAGENT_NAME),
                None,
            )
            if agent:
                DATAAGENT_ID = agent["id"]

if not DATAAGENT_ID:
    raise ValueError(f"Data Agent '{DATAAGENT_NAME}' not found. Run Notebook 03 first.")

print(f"Workspace  : {WORKSPACE_ID}")
print(f"Data Agent : {DATAAGENT_NAME} (ID: {DATAAGENT_ID})")

---
## Helper: Query the Agent

Utility function to send queries and handle the response.

In [ ]:
# Track conversation for multi-turn
_conversation_id = None

def ask_agent(question, new_conversation=False):
    """
    Send a natural-language query to the Data Agent.
    
    Args:
        question: The question to ask
        new_conversation: If True, start a fresh conversation context
    
    Returns:
        The agent's response text
    """
    global _conversation_id
    
    if new_conversation:
        _conversation_id = None
    
    payload = {"userMessage": question}
    if _conversation_id:
        payload["conversationId"] = _conversation_id
    
    print(f"Question: {question}")
    print("Thinking...")
    
    resp = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/dataAgents/{DATAAGENT_ID}/query",
        json=payload,
    )
    
    if resp.status_code == 404:
        print("\nERROR: Agent query endpoint returned 404.")
        print("The agent may not be published yet. Please publish it in the Fabric portal:")
        print(f"  https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}")
        return None
    
    if not resp.ok:
        print(f"\nERROR: HTTP {resp.status_code}: {resp.text[:500]}")
        return None
    
    data = resp.json()
    
    # Update conversation ID for multi-turn
    _conversation_id = data.get("conversationId", _conversation_id)
    
    # Extract response text (handles multiple response formats)
    answer = (
        data.get("response")
        or data.get("answer")
        or data.get("message")
        or data.get("text")
    )
    
    # Handle OpenAI-style response format
    if not answer and "choices" in data:
        choices = data["choices"]
        if choices and isinstance(choices, list):
            answer = choices[0].get("message", {}).get("content", "")
    
    if answer:
        print(f"\nAnswer:\n{answer}")
    else:
        print(f"\nRaw response:\n{json.dumps(data, indent=2)}")
    
    # Show citations if present
    citations = data.get("citations", [])
    if citations:
        print(f"\nCitations: {len(citations)}")
        for c in citations:
            print(f"   - {c}")
    
    print("-" * 60)
    return answer

---
## Readiness Check

Verify the agent is published and responding before running demo queries.

In [ ]:
# Quick readiness probe
probe_resp = client.post(
    f"v1/workspaces/{WORKSPACE_ID}/dataAgents/{DATAAGENT_ID}/query",
    json={"userMessage": "ping"},
)

if probe_resp.status_code == 404:
    print("Agent is NOT queryable (404). It may still be in Draft state.")
    print(f"Publish it at: https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}")
elif probe_resp.ok:
    print("Agent is queryable and ready!")
else:
    print(f"Agent returned HTTP {probe_resp.status_code} — may still be warming up.")
    print("Waiting 30 seconds and retrying...")
    time.sleep(30)
    probe2 = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/dataAgents/{DATAAGENT_ID}/query",
        json={"userMessage": "ping"},
    )
    if probe2.ok:
        print("Agent is now ready!")
    else:
        print(f"Agent still returning HTTP {probe2.status_code}. May need more time.")

---
## Demo Queries

Run sample queries to demonstrate the end-to-end pipeline.

In [ ]:
# Query 1: Overview
ask_agent("How many customers are in the Customer360 table?", new_conversation=True)

In [ ]:
# Query 2: Segment analysis
ask_agent("What are the different customer segments and how many customers are in each?")

In [ ]:
# Query 3: Churn risk
ask_agent("Which customers have the highest churn risk? Show me the top 10.")

In [ ]:
# Query 4: Revenue insights
ask_agent("What is the average monthly revenue by customer segment?")

In [ ]:
# Query 5: Geographic analysis
ask_agent("Which states have the highest total lifetime value? Show top 5.")

In [ ]:
# Query 6: Complex analytical question
ask_agent(
    "Find Enterprise customers with churn risk above 0.7 and monthly revenue above 1500. "
    "How many are there and what is their average lifetime value?"
)

---
## Interactive Query

Use this cell to ask your own questions.

In [ ]:
# Replace with your own question
YOUR_QUESTION = "What trends do you see in the customer data?"

ask_agent(YOUR_QUESTION)

---
## Architecture Summary

```
customer360.csv
      |
      v
Lakehouse (Customer360Lakehouse)
   +-- Customer360 table (Delta)
          |
          v
Semantic Model (auto-created from Lakehouse)
          |
          v
Ontology (Customer360Ontology)
   +-- Customer entity (bound to Customer360 table)
          |
          v
Data Agent (Customer360Agent)
   +-- Semantic Model as data source
   +-- Ontology attached
          |
          v
Natural Language Queries --> Answers
```